# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Diovalda22/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)
*Classification, clustering, ranking, or scoring — which one, and why?*

**Selected Lane:** Content Opportunity Scoring

**ML Task Type:** Ranking & Scoring supported by Binary classification for estimation probability

**Why This Type?**
The primary goal of this lane is not merely to assign a yes/no label (binary classification) or create groups (clustering), but rather to generate a **ranked priority queue** or a continuous opportunity score (e.g., 0–100) for each web page.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

print("Lane: Content Opportunity Scoring")
print("ML Task Type: Ranking / Priority Scoring (via Binary Classification Probability)")
print("Primary Output: Priority-ordered queue for content refresh team")


Lane: Content Opportunity Scoring
ML Task Type: Ranking / Priority Scoring (via Binary Classification Probability)
Primary Output: Priority-ordered queue for content refresh team


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target Variable:** `is_declining_label` (1 = declining search traffic, 0 = stable/growing).

**Label Source (Observed vs. Defined):**
This label is derived from an **observed search performance outcome**, specifically when `trend_direction == 'down'` (a comparison of the impression volume from the last 30 days versus the preceding 30 days).

*Important Note (Leakage Guard):* Per FlyRank's data rules, because the `trend_direction` and `trend_pct` columns are used to construct this target label, they **MUST NEVER** be included as input features for the model.

**Proxy Aspect:**
The Opportunity Score itself is a *proxy metric* that combines:
1. Observed Probability of Decline: $P(\text{decline} \mid X)$
2. Business Impact Floor / Traffic Volume: $\log(\text{impressions\_90d})$

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Load starter data
file_path = '../../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(file_path):
    file_path = 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(file_path)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("Target Column: is_declining_label (Observed outcome)")
print(df['is_declining_label'].value_counts(dropna=False, normalize=True).rename({1: 'Declining (1)', 0: 'Non-declining (0)'}))


Target Column: is_declining_label (Observed outcome)
is_declining_label
Declining (1)        0.542067
Non-declining (0)    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Evaluation & Business Metric:** **Precision@K** (specifically **Precision@100** and **Precision@500**).

**Why Precision@K?**
The editorial team has a limited working capacity (e.g., they might only be able to review 100 pages per week). Precision@100 answers the question: *"Out of the top 100 pages recommended by the model, what percentage are genuinely high-risk (declining) pages that actually need a refresh?"*
Maximizing Precision@K directly minimizes false positives, preventing the team from wasting valuable time revising pages that are actually still healthy.

**Secondary ML Metrics:**
- **ROC-AUC** & **PR-AUC**: To evaluate the overall classification and ranking quality across all possible thresholds.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(y_true, y_score, k=100):
    top_k_indices = np.argsort(y_score)[::-1][:k]
    return np.mean(y_true.iloc[top_k_indices] == 1)

# Hitung baseline sederhana menggunakan impresi mentah sebagai skor prioritas awal
baseline_p100 = precision_at_k(df['is_declining_label'], df['impressions_90d'], k=100)
baseline_p500 = precision_at_k(df['is_declining_label'], df['impressions_90d'], k=500)

print(f"Naive Baseline (Raw Impressions) Precision@100: {baseline_p100:.2%}")
print(f"Naive Baseline (Raw Impressions) Precision@500: {baseline_p500:.2%}")


Naive Baseline (Raw Impressions) Precision@100: 38.00%
Naive Baseline (Raw Impressions) Precision@500: 42.00%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
<br>
**Unit of Analysis:** **1 Row = 1 Pseudonymized Content Item (URL/Halaman)** per klien dalam jendela waktu 90 hari (`content_id` per `client_id`).

**Dataframe Slice:** 30.000 item konten dari 32 klien.
- **Identifiers:** `content_id`, `client_id` (hanya untuk grouping/split, bukan feature)
- **Features:** `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `content_age_days`, `word_count`, `engagement_rate`
- **Target/Proxy:** `is_declining_label` dan kandidat `opportunity_score_sketch`


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unit_cols = [
    'content_id', 'client_id', 'content_type', 
    'impressions_90d', 'clicks_90d', 'avg_position', 
    'ctr', 'engagement_rate', 'is_declining_label'
]

unit_df = df[unit_cols].copy()

# Buat sketsa Opportunity Score sederhana (Log Impressions * Decline Label)
unit_df['opportunity_score_sketch'] = np.log1p(unit_df['impressions_90d']) * unit_df['is_declining_label']

print(f"Dataframe Shape: {unit_df.shape[0]:,} baris × {unit_df.shape[1]} kolom")
print("Unit of Analysis: 1 Baris = 1 Pseudonymized Content Item")
unit_df.head(5)


Dataframe Shape: 30,000 baris × 10 kolom
Unit of Analysis: 1 Baris = 1 Pseudonymized Content Item


,content_id,client_id,content_type,impressions_90d,clicks_90d,avg_position,ctr,engagement_rate,is_declining_label,opportunity_score_sketch
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,10.6,0.76,5.88,1,8.243808
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,20.3,0.05,0.00,1,9.636980
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,36.5,0.09,0.00,1,9.440023
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,6.2,0.49,1.28,0,0.000000
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,44.0,0.13,0.00,1,9.859588


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML Beats a Fixed Static Rule:**

1. **Complex Non-Linear Signal Interactions:** The risk of traffic decay is driven by non-linear combinations of multiple variables, such as content age, CTR relative to average position, engagement rate, scroll depth, and AI traffic percentage.
2. **Varying Client & Content Characteristics:** Rigid rules like `IF impressions > 500 AND avg_position > 10` apply a one-size-fits-all threshold across all clients and content types (e.g., keyword articles vs. feedly articles). Because performance baselines differ per client, static rules generate too many false positives on large sites while completely missing decay on smaller, niche sites.
3. **The Need for Continuous Priority Ranking:** An `IF` rule only groups pages into binary buckets ("needs refresh" vs. "safe"). It cannot provide a precise, continuous priority ranking that aligns with the editorial team's limited weekly capacity. An ML model can learn continuous weights from dozens of signals to dynamically rank the review queue from most urgent to least urgent.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fixed_rule_mask = (df['impressions_90d'] >= 500) & (df['avg_position'] > 10)
flagged_by_rule = fixed_rule_mask.sum()
true_declining_in_rule = df[fixed_rule_mask]['is_declining_label'].sum()
rule_precision = true_declining_in_rule / flagged_by_rule if flagged_by_rule > 0 else 0

print(f"Fixed Rule ('impressions >= 500 & avg_position > 10') mem-flag {flagged_by_rule:,} halaman.")
print(f"Precision dari Fixed Rule: {rule_precision:.2%}")
print("Keterbatasan: Fixed rule tidak memberikan peringkat prioritas antar halaman yang di-flag dan melewatkan halaman ber-CTR tinggi yang sedang decline.")


Fixed Rule ('impressions >= 500 & avg_position > 10') mem-flag 9,162 halaman.
Precision dari Fixed Rule: 60.11%
Keterbatasan: Fixed rule tidak memberikan peringkat prioritas antar halaman yang di-flag dan melewatkan halaman ber-CTR tinggi yang sedang decline.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.